# Structure of the legacy securities lending data

Source table `crp_sftds_ecb_legacy.state_sl_079`, the trade states before June 2026, to be used from 2021-01-01 to 2026-05-31. Same ESMA report as the new table, stored the old way, with the collateral as extra rows per piece via `local_index` instead of arrays. Five checks before the cleaning query is mapped onto it. Single day checks use 2026-05-29, the last business day of May 2026, and rely on the table being partitioned by `business_date`.

In [11]:
import pyodbc
import pandas as pd
import numpy as np

cnxn = pyodbc.connect('DSN=Hermes_DSN',autocommit=True)
cursor = cnxn.cursor()
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 250)

Which column is the table partitioned by? Single day filters are only cheap if it is `business_date`.

In [2]:
query = f"""

SHOW PARTITIONS crp_sftds_ecb_legacy.state_sl_079

"""
df = pd.read_sql_query(query, cnxn)
df.tail(3)

C:\Users\hermesf\AppData\Local\Temp\ipykernel_19220\591883660.py:6: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, cnxn)


,db_type,business_date,tec_execution_date,#Rows,#Files,Size,Bytes Cached,Cache Replication,Format,Incremental stats,Location,EC Policy
1562,ECB,2026-07-22,20260728181837,5862581,20,2.44GB,NOT CACHED,NOT CACHED,PARQUET,true,s3a://devo-crp-2gn5maj9/sftds_ecb_legacy/db/st...,NONE
1563,ECB,2026-07-23,20260728211700,6053480,20,2.50GB,NOT CACHED,NOT CACHED,PARQUET,true,s3a://devo-crp-2gn5maj9/sftds_ecb_legacy/db/st...,NONE
1564,Total,,,40817302,25822,2.70TB,0B,,,,,


Answer. Partitioned by `db_type`, `business_date` and `tec_execution_date`, so single day filters read one day only. The partitions run past May 2026 into July, so the union with the new table has to cut at 2026-05-31.

## 1. What is a row, and what identifies a report?

Every report should have a row with `local_index = 0`, exactly one, and `techrcrdid` should be the same on all rows of a report.

In [3]:
query = f"""

SELECT local_index, COUNT(*) AS n
FROM crp_sftds_ecb_legacy.state_sl_079
WHERE business_date = '2026-05-29'
GROUP BY 1
ORDER BY 1
LIMIT 25

"""
df = pd.read_sql_query(query, cnxn)
df

C:\Users\hermesf\AppData\Local\Temp\ipykernel_19220\4001327716.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, cnxn)


,local_index,n
0,0,2159306
1,1,324573
2,2,90592
3,3,85450
4,4,68598
5,5,63294
6,6,60923
7,7,59094
8,8,57153
9,9,56395


In [4]:
query = f"""

SELECT COUNT(*) AS n_rows,
       COUNT(DISTINCT techrcrdid) AS n_reports,
       SUM(CASE WHEN local_index = 0 THEN 1 ELSE 0 END) AS n_row0,
       COUNT(DISTINCT CASE WHEN local_index = 0 THEN techrcrdid END) AS n_reports_with_row0,
       COUNT(DISTINCT tec_surrogate_key) AS n_surrogate_keys
FROM crp_sftds_ecb_legacy.state_sl_079
WHERE business_date = '2026-05-29'

"""
df = pd.read_sql_query(query, cnxn)
df

C:\Users\hermesf\AppData\Local\Temp\ipykernel_19220\1334257864.py:12: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, cnxn)


,n_rows,n_reports,n_row0,n_reports_with_row0,n_surrogate_keys
0,5631954,2159203,2159306,2159203,5631954


Answer. One row per report and collateral piece, about 2.2 million reports in 5.6 million rows on the day. Row 0 exists once per report, apart from about a hundred duplicates, `techrcrdid` identifies the report across its rows, and `tec_surrogate_key` is per row.

## 2. Which codes do the categorical fields use?

`direction` replaces `counterparty_side`, `is_opn_term` replaces `term_type`, `uncollsd` is a string where the new table has a boolean.

In [5]:
query = f"""

SELECT * FROM (
  SELECT 'direction' AS col, CAST(direction AS STRING) AS value, COUNT(*) AS n
  FROM crp_sftds_ecb_legacy.state_sl_079 WHERE business_date = '2026-05-29' AND local_index = 0 GROUP BY 1, 2
  UNION ALL
  SELECT 'is_opn_term' AS col, CAST(is_opn_term AS STRING) AS value, COUNT(*) AS n
  FROM crp_sftds_ecb_legacy.state_sl_079 WHERE business_date = '2026-05-29' AND local_index = 0 GROUP BY 1, 2
  UNION ALL
  SELECT 'uncollsd' AS col, CAST(uncollsd AS STRING) AS value, COUNT(*) AS n
  FROM crp_sftds_ecb_legacy.state_sl_079 WHERE business_date = '2026-05-29' AND local_index = 0 GROUP BY 1, 2
  UNION ALL
  SELECT 'rbtrate_type' AS col, CAST(rbtrate_type AS STRING) AS value, COUNT(*) AS n
  FROM crp_sftds_ecb_legacy.state_sl_079 WHERE business_date = '2026-05-29' AND local_index = 0 GROUP BY 1, 2
  UNION ALL
  SELECT 'ctrctmod_lvl' AS col, CAST(ctrctmod_lvl AS STRING) AS value, COUNT(*) AS n
  FROM crp_sftds_ecb_legacy.state_sl_079 WHERE business_date = '2026-05-29' AND local_index = 0 GROUP BY 1, 2
  UNION ALL
  SELECT 'ctrctmod_actntp' AS col, CAST(ctrctmod_actntp AS STRING) AS value, COUNT(*) AS n
  FROM crp_sftds_ecb_legacy.state_sl_079 WHERE business_date = '2026-05-29' AND local_index = 0 GROUP BY 1, 2
) u
ORDER BY col, n DESC

"""
df = pd.read_sql_query(query, cnxn)
df

C:\Users\hermesf\AppData\Local\Temp\ipykernel_19220\621492620.py:25: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, cnxn)


,col,value,n
0,ctrctmod_actntp,VALU,957670
1,ctrctmod_actntp,MODI,754049
2,ctrctmod_actntp,NEWT,265099
3,ctrctmod_actntp,COLU,175209
4,ctrctmod_actntp,CORR,7279
5,ctrctmod_lvl,TCTN,2024099
6,ctrctmod_lvl,None,131845
7,ctrctmod_lvl,PSTN,3362
8,direction,GIVE,1230279
9,direction,TAKE,797182


Answer. Action type, level and direction use the same codes as the new table. `is_opn_term` is '1' or '0', `rbtrate_type` is Fxd or Fltg instead of Fixed or Floating, and `uncollsd` is NORE or empty where the new table has a boolean. Which codes does the new table use for the term, so the legacy values can be mapped onto them?

In [ ]:
query = f"""

SELECT term_type, termination_optionality, COUNT(*) AS n
FROM xlab_ecb_prj_sftds_cb_common.hermesf_sl
WHERE reference_period = (SELECT MAX(reference_period) FROM xlab_ecb_prj_sftds_cb_common.hermesf_sl)
GROUP BY 1, 2
ORDER BY n DESC

"""
df = pd.read_sql_query(query, cnxn)
df

## 3. How are quantity and price stored?

The legacy table splits units and nominal, and monetary, percentage and yield prices, into separate columns. The notation type of the new table has to be derived from which column is filled, using the new table's codes.

In [6]:
query = f"""

SELECT CASE WHEN lndata_assttp_scty_qty IS NOT NULL THEN 1 ELSE 0 END AS has_units,
       CASE WHEN lndata_assttp_scty_nmnl_amt IS NOT NULL THEN 1 ELSE 0 END AS has_nominal,
       CASE WHEN lndata_assttp_scty_unitpric_amt IS NOT NULL THEN 1 ELSE 0 END AS has_price_amount,
       CASE WHEN lndata_assttp_scty_unitpric_pctg IS NOT NULL THEN 1 ELSE 0 END AS has_price_pct,
       CASE WHEN lndata_assttp_scty_unitpric_yld IS NOT NULL THEN 1 ELSE 0 END AS has_price_yield,
       COUNT(*) AS n
FROM crp_sftds_ecb_legacy.state_sl_079
WHERE business_date = '2026-05-29' AND local_index = 0
GROUP BY 1, 2, 3, 4, 5
ORDER BY n DESC

"""
df = pd.read_sql_query(query, cnxn)
df

C:\Users\hermesf\AppData\Local\Temp\ipykernel_19220\2545733726.py:15: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, cnxn)


,has_units,has_nominal,has_price_amount,has_price_pct,has_price_yield,n
0,1,0,1,0,0,1429108
1,0,1,0,1,0,507417
2,0,0,0,0,0,131995
3,0,1,1,0,0,85005
4,1,0,0,1,0,4150
5,0,0,1,0,0,1596
6,0,0,0,1,0,35


The codes the new table uses for the same distinction.

In [7]:
query = f"""

SELECT loan_security_quantity_or_nominal_amount_notation_type AS quantity_notation,
       loan_security_price_notation_type AS price_notation,
       COUNT(*) AS n
FROM crp_sftds_ecb.trade_states_securitieslending
WHERE reference_period = (SELECT MAX(reference_period) FROM crp_sftds_ecb.trade_states_securitieslending)
GROUP BY 1, 2
ORDER BY n DESC

"""
df = pd.read_sql_query(query, cnxn)
df

C:\Users\hermesf\AppData\Local\Temp\ipykernel_19220\57967840.py:12: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, cnxn)


,quantity_notation,price_notation,n
0,Quantity,Monetary value,1515437
1,Monetary value,Percentage,565627
2,None,None,133420
3,Monetary value,Monetary value,93643
4,Quantity,Percentage,6537
5,None,Monetary value,1593
6,None,Percentage,35


Answer. Same shape on both sides. Units go with a monetary price, a nominal with a percentage price. For the legacy rows the codes are derived, `Quantity` when the units column is filled and `Monetary value` when the nominal is, and `Monetary value` or `Percentage` for the price in the same way.

## 4. Are the collateral component counts per report?

`collcmpnttp_scty` and `collcmpnttp_csh` should equal the number of rows of a report that carry a security or a cash piece.

In [8]:
query = f"""

SELECT r.collcmpnttp_scty, r.n_sec_rows, r.collcmpnttp_csh, r.n_cash_rows, COUNT(*) AS n_reports
FROM (
  SELECT techrcrdid,
         MAX(collcmpnttp_scty) AS collcmpnttp_scty,
         MAX(collcmpnttp_csh) AS collcmpnttp_csh,
         SUM(CASE WHEN assttp_scty_id IS NOT NULL THEN 1 ELSE 0 END) AS n_sec_rows,
         SUM(CASE WHEN assttp_csh_amt IS NOT NULL THEN 1 ELSE 0 END) AS n_cash_rows
  FROM crp_sftds_ecb_legacy.state_sl_079
  WHERE business_date = '2026-05-29'
  GROUP BY techrcrdid
) r
GROUP BY 1, 2, 3, 4
ORDER BY n_reports DESC
LIMIT 30

"""
df = pd.read_sql_query(query, cnxn)
df

C:\Users\hermesf\AppData\Local\Temp\ipykernel_19220\1641786653.py:19: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, cnxn)


,collcmpnttp_scty,n_sec_rows,collcmpnttp_csh,n_cash_rows,n_reports
0,0,0,0,0,1606036
1,0,0,1,1,325142
2,0,0,1,2,65847
3,1,1,0,0,48574
4,1,2,0,0,22778
5,1,4,0,0,16483
6,1,16,0,0,15923
7,1,5,0,0,5211
8,1,3,0,0,4091
9,1,55,0,0,2488


Answer. No, they are flags. `collcmpnttp_scty` and `collcmpnttp_csh` are 0 or 1, so the number of pieces has to be counted from the rows per `techrcrdid`. The reports with 55 to 62 securities pieces are the pool rows.

## 5. Does the deduplication key behave as in the new table?

Same group check as in the data structure notebook, on the row 0 rows only so that collateral rows do not count as legs, and on one day per year, the last business day of May from 2021 to 2026, instead of the full five years.

In [9]:
query = f"""

SELECT n_rows, n_best, n_dates, COUNT(*) AS n_groups
FROM (
  SELECT ruti, business_date,
         COUNT(*) AS n_rows,
         SUM(CASE WHEN best_value_leg = 1 THEN 1 ELSE 0 END) AS n_best,
         COUNT(DISTINCT evtdt) AS n_dates
  FROM crp_sftds_ecb_legacy.state_sl_079
  WHERE business_date IN ('2021-05-31', '2022-05-31', '2023-05-31', '2024-05-31', '2025-05-30', '2026-05-29')
    AND local_index = 0
  GROUP BY ruti, business_date
) g
GROUP BY n_rows, n_best, n_dates
ORDER BY n_groups DESC

"""
df = pd.read_sql_query(query, cnxn)
df.head(30)

C:\Users\hermesf\AppData\Local\Temp\ipykernel_19220\2601730562.py:18: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, cnxn)


,n_rows,n_best,n_dates,n_groups
0,1,0,1,11081913
1,2,1,1,672065
2,2,1,2,152828
3,2,0,2,112836
4,2,0,1,77425
5,1,1,1,6
6,3,0,2,4
7,3,1,2,2
8,30624,0,483,1
9,22427,0,230,1


Answer. Mostly as in the new table. Single legs, pairs with one flagged leg, and one group without `ruti` per day. New here are about 190 thousand pairs on the six days, a fifth of all pairs, with no best value leg flag at all. For those the timestamp tie break decides. How does the flag relate to `ruti_paired`?

In [10]:
query = f"""

SELECT ruti_count, ruti_paired, best_value_leg, COUNT(*) AS n
FROM crp_sftds_ecb_legacy.state_sl_079
WHERE business_date = '2026-05-29' AND local_index = 0
  AND ruti IS NOT NULL AND ruti <> ''
GROUP BY 1, 2, 3
ORDER BY 1, 2, 3

"""
df = pd.read_sql_query(query, cnxn)
df

C:\Users\hermesf\AppData\Local\Temp\ipykernel_19220\2995275988.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, cnxn)


,ruti_count,ruti_paired,best_value_leg,n
0,1,0,NaN,1625973
1,2,1,0.0,200743
2,2,1,1.0,200745


Answer. On this day every pair has exactly one flagged leg and `ruti_paired` is 1 for all of them, so the unflagged pairs come from other days. Which ones?

In [2]:
query = f"""

SELECT business_date, n_best, COUNT(*) AS n_pairs
FROM (
  SELECT ruti, business_date,
         COUNT(*) AS n_rows,
         SUM(CASE WHEN best_value_leg = 1 THEN 1 ELSE 0 END) AS n_best
  FROM crp_sftds_ecb_legacy.state_sl_079
  WHERE business_date IN ('2021-05-31', '2022-05-31', '2023-05-31', '2024-05-31', '2025-05-30', '2026-05-29')
    AND local_index = 0 AND ruti IS NOT NULL AND ruti <> ''
  GROUP BY ruti, business_date
) g
WHERE n_rows = 2
GROUP BY 1, 2
ORDER BY 1, 2

"""
df = pd.read_sql_query(query, cnxn)
df

C:\Users\hermesf\AppData\Local\Temp\ipykernel_24864\1870964915.py:18: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, cnxn)


,business_date,n_best,n_pairs
0,2021-05-31,0,23548
1,2021-05-31,1,70757
2,2022-05-31,0,23862
3,2022-05-31,1,120945
4,2023-05-31,0,18554
5,2023-05-31,1,138817
6,2024-05-31,0,32683
7,2024-05-31,1,140802
8,2025-05-30,0,39122
9,2025-05-30,1,152829


The rows without `ruti` should again be the net exposure collateral updates without UTI.

In [11]:
query = f"""

SELECT ctrctmod_actntp, CASE WHEN uti IS NULL THEN 1 ELSE 0 END AS uti_missing, COUNT(*) AS n
FROM crp_sftds_ecb_legacy.state_sl_079
WHERE business_date = '2026-05-29' AND local_index = 0
  AND (ruti IS NULL OR ruti = '')
GROUP BY 1, 2
ORDER BY n DESC

"""
df = pd.read_sql_query(query, cnxn)
df

C:\Users\hermesf\AppData\Local\Temp\ipykernel_19220\3488041391.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, cnxn)


,ctrctmod_actntp,uti_missing,n
0,COLU,1,131845


Answer. Yes, all of them are collateral updates without UTI, 131,845 on the day, the same number as the group without `ruti` in the check above.

## 6. Why do the early months have more rows than expected?

January and February 2021 load with about 50 million rows, 2.5 million loans per day against 1.83 million on the latest day of the new table. Either there were more trades, or more trades appear twice because their two legs carry different UTIs and therefore different `ruti` values. Rows per day first, to see whether the level is uniform or driven by single days.

In [3]:
query = f"""

SELECT reference_period, COUNT(*) AS n
FROM xlab_ecb_prj_sftds_cb_common.hermesf_sl_legacy
WHERE reference_period BETWEEN CAST('2021-01-01' AS DATE) AND CAST('2021-01-31' AS DATE)
GROUP BY 1
ORDER BY 1

"""
df = pd.read_sql_query(query, cnxn)
df

C:\Users\hermesf\AppData\Local\Temp\ipykernel_24864\2865827688.py:10: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, cnxn)


,reference_period,n
0,2021-01-01,2360102
1,2021-01-04,2430190
2,2021-01-05,2452413
3,2021-01-06,2471556
4,2021-01-07,2482181
5,2021-01-08,2507602
6,2021-01-11,2513199
7,2021-01-12,2544876
8,2021-01-13,2574097
9,2021-01-14,2593264


Answer. The level is uniform, 2.4 to 2.8 million loans per day, rising steadily through the month, which fits the usual rebuild of lending balances after the year end. Two weekdays are missing, 21 and 29 January, so the legacy partitions have gaps. Checked further below.

Is a January day loaded once? Row 0 rows should equal reports, as on 2026-05-29.

In [4]:
query = f"""

SELECT COUNT(*) AS n_row0,
       COUNT(DISTINCT techrcrdid) AS n_reports,
       COUNT(DISTINCT tec_execution_date) AS n_executions
FROM crp_sftds_ecb_legacy.state_sl_079
WHERE business_date = '2021-01-15' AND local_index = 0

"""
df = pd.read_sql_query(query, cnxn)
df

C:\Users\hermesf\AppData\Local\Temp\ipykernel_24864\799725447.py:10: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, cnxn)


,n_row0,n_reports,n_executions
0,2800309,2796618,1


Answer. One execution and row 0 rows equal to reports apart from a tenth of a percent, so the day is loaded once and duplicates are ruled out.

Singles versus pairs on a January day compared with 2026-05-29. A higher share of singles in 2021, with a similar GIVE and TAKE mix among them, points at legs that did not pair rather than at more trades.

In [5]:
query = f"""

SELECT business_date, n_rows,
       SUM(CASE WHEN direction = 'GIVE' THEN 1 ELSE 0 END) AS n_give,
       SUM(CASE WHEN direction = 'TAKE' THEN 1 ELSE 0 END) AS n_take,
       COUNT(*) AS n_legs
FROM (
  SELECT business_date, direction,
         COUNT(*) OVER (PARTITION BY ruti, business_date) AS n_rows
  FROM crp_sftds_ecb_legacy.state_sl_079
  WHERE business_date IN ('2021-01-15', '2026-05-29') AND local_index = 0
    AND ruti IS NOT NULL AND ruti <> ''
) g
GROUP BY 1, 2
ORDER BY 1, 2

"""
df = pd.read_sql_query(query, cnxn)
df

C:\Users\hermesf\AppData\Local\Temp\ipykernel_24864\1433398638.py:18: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, cnxn)


,business_date,n_rows,n_give,n_take,n_legs
0,2021-01-15,1,1766200,883787,2649987
1,2021-01-15,2,62888,65514,128402
2,2021-01-15,3,8,4,12
3,2026-05-29,1,976998,543993,1520991
4,2026-05-29,2,253281,253189,506470


Answer. On 2021-01-15 only 64 thousand trades are pairs against 2.65 million singles, a pairing share of 2.4 percent. On 2026-05-29 it is 253 thousand pairs against 1.52 million singles, 14 percent. The GIVE and TAKE mix among the singles is the same in both years, two thirds borrower reports. So 2021 has far more single legs. Whether they are unpaired double reports shows in the UTIs. Two single legs of the same trade share a UTI even when their `ruti` differs, for instance because one side reported a branch LEI of the other.

In [ ]:
query = f"""

SELECT business_date, n_legs_same_uti, COUNT(*) AS n_utis
FROM (
  SELECT business_date, uti, COUNT(*) AS n_legs_same_uti
  FROM (
    SELECT business_date, uti,
           COUNT(*) OVER (PARTITION BY ruti, business_date) AS n_rows
    FROM crp_sftds_ecb_legacy.state_sl_079
    WHERE business_date IN ('2021-01-15', '2026-05-29') AND local_index = 0
      AND ruti IS NOT NULL AND ruti <> ''
  ) x
  WHERE n_rows = 1
  GROUP BY 1, 2
) u
GROUP BY 1, 2
ORDER BY 1, 2

"""
df = pd.read_sql_query(query, cnxn)
df

Answer. Twins are negligible in both years, about a thousand UTIs out of 2.6 million and 1.5 million. The extra single legs of 2021 are distinct UTIs, so they are not pairs that missed each other on the `ruti`. That leaves a bigger market, double reports with different UTIs on each leg, or loans that were never reported as terminated and keep sitting in the state. The two checks below separate them.

Which weekdays are missing from the legacy partitions between 2021-01-01 and 2026-05-31?

In [ ]:
part = pd.read_sql_query("SHOW PARTITIONS crp_sftds_ecb_legacy.state_sl_079", cnxn)
have = pd.to_datetime(part['business_date'], errors='coerce').dropna()
missing = pd.bdate_range('2021-01-01', '2026-05-31').difference(have)
print(len(missing), 'weekdays missing')
missing

Answer. Eight weekdays are missing, all in 2021, two in January, two in February, one in June and three in November. The series has these holes and nothing can fill them.

How stale are the loan states? A live loan gets a valuation update almost daily, so a last event far in the past marks a loan that was probably returned without a termination report. Compared on a day of each period, on the cleaned tables.

In [9]:
query = f"""

SELECT * FROM (
  SELECT 'legacy 2021-01-15' AS day,
         CASE WHEN DATEDIFF(CAST(reference_period AS TIMESTAMP), CAST(event_date AS TIMESTAMP)) <= 7 THEN 'a. up to 7 days'
              WHEN DATEDIFF(CAST(reference_period AS TIMESTAMP), CAST(event_date AS TIMESTAMP)) <= 30 THEN 'b. 8 to 30 days'
              WHEN DATEDIFF(CAST(reference_period AS TIMESTAMP), CAST(event_date AS TIMESTAMP)) <= 90 THEN 'c. 31 to 90 days'
              ELSE 'd. over 90 days' END AS last_event_age,
         COUNT(*) AS n
  FROM xlab_ecb_prj_sftds_cb_common.hermesf_sl_legacy
  WHERE reference_period = CAST('2021-01-15' AS DATE)
  GROUP BY 1, 2
  UNION ALL
  SELECT 'new 2026-06-15',
         CASE WHEN DATEDIFF(CAST(reference_period AS TIMESTAMP), CAST(event_date AS TIMESTAMP)) <= 7 THEN 'a. up to 7 days'
              WHEN DATEDIFF(CAST(reference_period AS TIMESTAMP), CAST(event_date AS TIMESTAMP)) <= 30 THEN 'b. 8 to 30 days'
              WHEN DATEDIFF(CAST(reference_period AS TIMESTAMP), CAST(event_date AS TIMESTAMP)) <= 90 THEN 'c. 31 to 90 days'
              ELSE 'd. over 90 days' END,
         COUNT(*)
  FROM xlab_ecb_prj_sftds_cb_common.hermesf_sl_new
  WHERE reference_period = CAST('2026-06-15' AS DATE)
  GROUP BY 1, 2
) u
ORDER BY day, last_event_age

"""
df = pd.read_sql_query(query, cnxn)
df

C:\Users\hermesf\AppData\Local\Temp\ipykernel_24864\1797953537.py:27: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, cnxn)


,day,last_event_age,n
0,legacy 2021-01-15,a. up to 7 days,668032
1,legacy 2021-01-15,b. 8 to 30 days,251932
2,legacy 2021-01-15,c. 31 to 90 days,662864
3,legacy 2021-01-15,d. over 90 days,1015954
4,new 2026-06-15,a. up to 7 days,1452860
5,new 2026-06-15,b. 8 to 30 days,28544
6,new 2026-06-15,c. 31 to 90 days,35019
7,new 2026-06-15,d. over 90 days,233548


Answer. This is the explanation. On 2026-06-15, 83 percent of the loans had an event within a week and 13 percent none for over 90 days. On 2021-01-15 only 26 percent had an event within a week, a quarter had their last one 31 to 90 days back and 39 percent, a million loans, over 90 days back. Counting only loans with an event in the last 90 days, the two days are close, 1.58 million against 1.52 million. So the higher 2021 level consists of states nobody maintained, loans returned without a termination report and loans whose reporters did not send valuation updates. Both kinds carry stale values.

Are there double reports with different UTIs? A borrower reported loan and a lender reported loan on the same day with the same lender, borrower, ISIN, quantity and start date are almost certainly one trade. Counted as borrower rows that have at least one such lender match.

In [10]:
query = f"""

SELECT * FROM (
  SELECT 'legacy 2021-01-15' AS day, COUNT(DISTINCT b.tec_surrogate_key) AS n_borrower_rows_with_lender_twin
  FROM xlab_ecb_prj_sftds_cb_common.hermesf_sl_legacy b
  JOIN xlab_ecb_prj_sftds_cb_common.hermesf_sl_legacy l
    ON l.reference_period = b.reference_period AND l.lender_id = b.lender_id AND l.borrower_id = b.borrower_id
   AND l.isin = b.isin AND l.loan_quantity = b.loan_quantity AND l.start_date = b.start_date
  WHERE b.reference_period = CAST('2021-01-15' AS DATE)
    AND b.reported_by = 'borrower' AND l.reported_by = 'lender'
  UNION ALL
  SELECT 'new 2026-06-15', COUNT(DISTINCT b.tec_surrogate_key)
  FROM xlab_ecb_prj_sftds_cb_common.hermesf_sl_new b
  JOIN xlab_ecb_prj_sftds_cb_common.hermesf_sl_new l
    ON l.reference_period = b.reference_period AND l.lender_id = b.lender_id AND l.borrower_id = b.borrower_id
   AND l.isin = b.isin AND l.loan_quantity = b.loan_quantity AND l.start_date = b.start_date
  WHERE b.reference_period = CAST('2026-06-15' AS DATE)
    AND b.reported_by = 'borrower' AND l.reported_by = 'lender'
) u
ORDER BY day

"""
df = pd.read_sql_query(query, cnxn)
df

C:\Users\hermesf\AppData\Local\Temp\ipykernel_24864\153687272.py:23: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, cnxn)


,day,n_borrower_rows_with_lender_twin
0,legacy 2021-01-15,2782
1,new 2026-06-15,3823


Answer. No. About three thousand such twins on either day, a tenth of a percent. Double reports with different UTIs are not the explanation.

## 7. Why are the volumes far too high?

The raw series reaches 12 trillion EUR in 2023, an order of magnitude above the size of the market. Is the deduplication at fault? In the final table every trade and day must appear exactly once.

In [14]:
query = f"""

SELECT n_rows_per_trade, COUNT(*) AS n_trades
FROM (
  SELECT tec_ruti, reference_period, COUNT(*) AS n_rows_per_trade
  FROM xlab_ecb_prj_sftds_cb_common.hermesf_sl
  WHERE reference_period IN (CAST('2023-06-15' AS DATE), CAST('2026-06-15' AS DATE))
  GROUP BY 1, 2
) g
GROUP BY 1
ORDER BY 1

"""
df = pd.read_sql_query(query, cnxn)
df

C:\Users\hermesf\AppData\Local\Temp\ipykernel_24864\724726457.py:14: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, cnxn)


,n_rows_per_trade,n_trades
0,1,3022208


Where does the volume sit? Loans by size, on a 2023 day and a 2026 day. A securities loan above one billion is already unusual, anything above ten billion is a reporting error.

In [15]:
query = f"""

SELECT reference_period,
       CASE WHEN loan_value_eur < 1e6  THEN 'a. below 1m'
            WHEN loan_value_eur < 1e7  THEN 'b. 1m to 10m'
            WHEN loan_value_eur < 1e8  THEN 'c. 10m to 100m'
            WHEN loan_value_eur < 1e9  THEN 'd. 100m to 1bn'
            WHEN loan_value_eur < 1e10 THEN 'e. 1bn to 10bn'
            ELSE 'f. over 10bn' END AS size_bucket,
       COUNT(*) AS n_loans,
       ROUND(SUM(loan_value_eur) / 1e9, 1) AS volume_bn_eur
FROM xlab_ecb_prj_sftds_cb_common.hermesf_sl
WHERE reference_period IN (CAST('2023-06-15' AS DATE), CAST('2026-06-15' AS DATE))
GROUP BY 1, 2
ORDER BY 1, 2

"""
df = pd.read_sql_query(query, cnxn)
df

C:\Users\hermesf\AppData\Local\Temp\ipykernel_24864\289568185.py:18: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, cnxn)


,reference_period,size_bucket,n_loans,volume_bn_eur
0,2023-06-15,a. below 1m,1081542,137.8
1,2023-06-15,b. 1m to 10m,140530,430.7
2,2023-06-15,c. 10m to 100m,41447,1289.8
3,2023-06-15,d. 100m to 1bn,6565,1894.8
4,2023-06-15,e. 1bn to 10bn,2083,6577.5
5,2023-06-15,f. over 10bn,70,1064.5
6,2026-06-15,a. below 1m,1494527,195.5
7,2026-06-15,b. 1m to 10m,197057,594.7
8,2026-06-15,c. 10m to 100m,53298,1578.1
9,2026-06-15,d. 100m to 1bn,4811,1050.3


The largest loans of the 2023 day, with the fields the value is built from.

In [16]:
query = f"""

SELECT reference_period, reported_by, lender_id, borrower_id, isin, loan_security_type,
       loan_quantity, loan_quantity_notation, loan_price, loan_price_notation, loan_price_currency,
       loan_value, loan_value_currency, loan_value_eur, loan_market_value_eur, collateral_type
FROM xlab_ecb_prj_sftds_cb_common.hermesf_sl
WHERE reference_period = CAST('2023-06-15' AS DATE)
ORDER BY loan_value_eur DESC
LIMIT 20

"""
df = pd.read_sql_query(query, cnxn)
df

C:\Users\hermesf\AppData\Local\Temp\ipykernel_24864\2240937598.py:12: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, cnxn)


,reference_period,reported_by,lender_id,borrower_id,isin,loan_security_type,loan_quantity,loan_quantity_notation,loan_price,loan_price_notation,loan_price_currency,loan_value,loan_value_currency,loan_value_eur,loan_market_value_eur,collateral_type
0,2023-06-15,lender,549300AWTWMSTGL2GG21,R0MUWSFPU8MPRO8K5P83,FR0010916924,GOVS,3.807027e+08,Monetary value,108.15800,Monetary value,EUR,4.117604e+10,EUR,4.117604e+10,4.117604e+10,basket
1,2023-06-15,borrower,C3GTMMZIHMY46P4OIX74,724500Q03K04L0479N30,JP1201571G68,GOVS,5.000000e+10,Monetary value,98.24084,Monetary value,JPY,4.912042e+12,JPY,3.199402e+10,3.197486e+08,none
2,2023-06-15,borrower,54930044PULMORCKB765,O2RNE8IBXP4R0TD8PU41,ES0000012B62,GOVS,3.200000e+08,Monetary value,98.55770,Monetary value,EUR,3.153846e+10,EUR,3.153846e+10,3.162912e+05,net_exposure
3,2023-06-15,borrower,549300WCGB70D06XZS54,529900UC2OD7II24Z667,CH0010645932,OEQU,9.041900e+06,Quantity,3458.86362,Monetary value,EUR,3.127470e+10,EUR,3.127470e+10,3.127470e+10,cash
4,2023-06-15,borrower,RR3QWICWWIPCS8A4S074,X3CZP3CK64YBHON1LE12,JP1201811N77,GOVS,5.000000e+10,Monetary value,94.39466,Monetary value,JPY,4.719733e+12,JPY,3.074144e+10,3.226289e+08,none
5,2023-06-15,borrower,2ZCNRR8UK83OBTEK2170,X3CZP3CK64YBHON1LE12,ES0312343017,SEPR,4.665626e+07,Monetary value,596.00000,Monetary value,EUR,2.780713e+10,EUR,2.780713e+10,6.430122e+06,none
6,2023-06-15,borrower,54930044PULMORCKB765,O2RNE8IBXP4R0TD8PU41,FR0014003513,GOVS,2.800000e+08,Monetary value,89.93000,Monetary value,EUR,2.518040e+10,EUR,2.518040e+10,2.555616e+08,net_exposure
7,2023-06-15,borrower,54930044PULMORCKB765,O2RNE8IBXP4R0TD8PU41,FR0014007TY9,GOVS,2.620000e+08,Monetary value,94.86000,Monetary value,EUR,2.485332e+10,EUR,2.485332e+10,2.504458e+08,net_exposure
8,2023-06-15,borrower,969500QKVPV2H8UXM738,1VUV7VQFKUOQSJ21A208,FR0011486067,GOVS,2.340000e+08,Monetary value,10419.00000,Percentage,None,2.438046e+10,EUR,2.438046e+10,2.438046e+08,net_exposure
9,2023-06-15,borrower,969500QKVPV2H8UXM738,1VUV7VQFKUOQSJ21A208,FR0011619436,GOVS,2.000000e+08,Monetary value,10728.00000,Percentage,None,2.145600e+10,EUR,2.145600e+10,2.145600e+08,net_exposure


Is the loan value consistent with quantity times price? For a percentage price the value should be nominal times price over 100. A ratio near 100 means the division was skipped, which inflates a bond loan a hundredfold.

In [17]:
query = f"""

SELECT reference_period, loan_price_notation,
       CASE WHEN ratio < 0.5 THEN 'a. below half'
            WHEN ratio < 2   THEN 'b. consistent'
            WHEN ratio < 50  THEN 'c. 2x to 50x'
            WHEN ratio < 200 THEN 'd. 50x to 200x'
            ELSE 'e. over 200x' END AS value_vs_quantity_x_price,
       COUNT(*) AS n_loans,
       ROUND(SUM(loan_value_eur) / 1e9, 1) AS volume_bn_eur
FROM (
  SELECT reference_period, loan_price_notation, loan_value_eur,
         loan_value / NULLIF(loan_quantity * loan_price
                             / CASE WHEN loan_price_notation = 'Percentage' THEN 100 ELSE 1 END, 0) AS ratio
  FROM xlab_ecb_prj_sftds_cb_common.hermesf_sl
  WHERE reference_period IN (CAST('2023-06-15' AS DATE), CAST('2026-06-15' AS DATE))
) x
GROUP BY 1, 2, 3
ORDER BY 1, 2, 3

"""
df = pd.read_sql_query(query, cnxn)
df

C:\Users\hermesf\AppData\Local\Temp\ipykernel_24864\2859462754.py:22: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, cnxn)


,reference_period,loan_price_notation,value_vs_quantity_x_price,n_loans,volume_bn_eur
0,2023-06-15,Monetary value,a. below half,128508,482.4
1,2023-06-15,Monetary value,b. consistent,903257,7256.8
2,2023-06-15,Monetary value,c. 2x to 50x,5203,12.6
3,2023-06-15,Monetary value,d. 50x to 200x,2040,284.0
4,2023-06-15,Monetary value,e. over 200x,3522,121.4
5,2023-06-15,Percentage,a. below half,1489,21.8
6,2023-06-15,Percentage,b. consistent,216819,3106.9
7,2023-06-15,Percentage,c. 2x to 50x,531,1.5
8,2023-06-15,Percentage,d. 50x to 200x,10462,97.0
9,2023-06-15,Percentage,e. over 200x,406,10.9


Is the EUR conversion right? The median of EUR value over original value should be the exchange rate, about 0.006 for JPY and about 0.9 for USD. A value of 1 for a foreign currency means the amount was never converted.

In [ ]:
query = f"""

SELECT reference_period, loan_value_currency,
       COUNT(*) AS n_loans,
       ROUND(SUM(loan_value_eur) / 1e9, 1) AS volume_bn_eur,
       ROUND(APPX_MEDIAN(loan_value_eur / NULLIF(loan_value, 0)), 4) AS median_eur_per_unit
FROM xlab_ecb_prj_sftds_cb_common.hermesf_sl
WHERE reference_period IN (CAST('2023-06-15' AS DATE), CAST('2026-06-15' AS DATE))
GROUP BY 1, 2
ORDER BY 1, volume_bn_eur DESC

"""
df = pd.read_sql_query(query, cnxn)
df.head(60)

C:\Users\hermesf\AppData\Local\Temp\ipykernel_24864\3429754314.py:13: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, cnxn)


,reference_period,loan_value_currency,n_loans,volume_bn_eur,median_eur_per_unit
0,2023-06-15,USD,418608,6554.8,0.9137
1,2023-06-15,EUR,411207,4067.4,1.0000
2,2023-06-15,GBP,53898,331.4,1.1684
3,2023-06-15,JPY,218491,296.2,0.0065
4,2023-06-15,KRW,13933,32.5,0.0007
5,2023-06-15,CHF,18483,27.8,1.0247
6,2023-06-15,CAD,26302,26.2,0.6911
7,2023-06-15,DKK,4300,12.8,0.1342
8,2023-06-15,AUD,8990,11.3,0.6294
9,2023-06-15,SEK,23708,9.2,0.0862
